# 1. Data Analysis

Librerias relevantes para el análisis

In [258]:
library(rio)
library(ggplot2)
library(caTools)
library(DescTools)
library(dplyr)
library(Hmisc)
library(psych)

In [259]:
url="https://raw.githubusercontent.com/d2cml-ai/CausalAI-Course/main/data/wage2015_subsample_inference.csv"
wage<-read.csv(url)

In [260]:
names(wage)

[1] "rownames" "wage"     "lwage"    "sex"      "shs"      "hsg"     
 [7] "scl"      "clg"      "ad"       "mw"       "so"       "we"      
[13] "ne"       "exp1"     "exp2"     "exp3"     "exp4"     "occ"     
[19] "occ2"     "ind"      "ind2"

2. Reporte de NAs


In [261]:
colSums(is.na(wage))

rownames     wage    lwage      sex      shs      hsg      scl      clg 
       0        0        0        0        0        0        0        0 
      ad       mw       so       we       ne     exp1     exp2     exp3 
       0        0        0        0        0        0        0        0 
    exp4      occ     occ2      ind     ind2 
       0        0        0        0        0

No existen datos perdidos en la base de datos

### 3.  Reporte de estadísticas descriptivas

In [290]:
describe(wage)

,vars,n,mean,sd,median,trimmed,mad,min,max,range,skew,kurtosis,se
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
rownames,1,5150,1.563635e+04,9.700744e+03,15260.000000,1.546166e+04,12622.115100,10.000000,3.264300e+04,32633.000000,0.14453716,-1.21638579,1.351766e+02
wage,2,5150,2.341041e+01,2.100302e+01,19.230769,2.053559e+01,10.264154,3.021978,5.288457e+02,525.823695,10.76152974,210.91873662,2.926699e-01
lwage,3,5150,2.970787e+00,5.703848e-01,2.956512,2.956854e+00,0.539897,1.105912,6.270697e+00,5.164785,0.37952616,1.14948916,7.948118e-03
sex,4,5150,4.444660e-01,4.969547e-01,0.000000,4.305825e-01,0.000000,0.000000,1.000000e+00,1.000000,0.22345378,-1.95044702,6.924894e-03
shs,5,5150,2.330097e-02,1.508723e-01,0.000000,0.000000e+00,0.000000,0.000000,1.000000e+00,1.000000,6.31801090,37.92462584,2.102354e-03
hsg,6,5150,2.438835e-01,4.294650e-01,0.000000,1.798544e-01,0.000000,0.000000,1.000000e+00,1.000000,1.19249205,-0.57807485,5.984448e-03
scl,7,5150,2.780583e-01,4.480858e-01,0.000000,2.225728e-01,0.000000,0.000000,1.000000e+00,1.000000,0.99042939,-1.01924743,6.243923e-03
clg,8,5150,3.176699e-01,4.656155e-01,0.000000,2.720874e-01,0.000000,0.000000,1.000000e+00,1.000000,0.78302667,-1.38713847,6.488194e-03
ad,9,5150,1.370874e-01,3.439730e-01,0.000000,4.635922e-02,0.000000,0.000000,1.000000e+00,1.000000,2.10971318,2.45136580,4.793146e-03


**Interpretación** : se puede extraer información relevante si nos enfocamos en la variable del salario del individuo. Se puede observar que el salario promedio de la muestra es de 23.41 dólares.No obstante, la desviación estándar es muy alta(21), lo cual quiere decir que existe una alta dispersión del salario con respecto al promedio. Por ejemplo, existe un individuo cuyo salario es de 528.85 dólares y otro cuyo salario es de 3.02 dólares. En aras de obtener información más estable sobre el salario se recurre a la mediana, la cual nos dice que el 50% de la muestra posee un salario entre los 3.02 y 19.23 dólares. Por último, la muestra presenta una asimetría positiva, lo cual implica que existe un mayor número de trabajadores con un salario menor al promedio.

4. Obtener el número de mujeres con un grado académico alto que poseen un salario del 25% más rico de la muestra

4.1 Identificación de cuantiles

In [263]:
summary(wage$wage)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  3.022  13.461  19.231  23.410  27.778 528.846 

4.2 Creamos nueva variable

In [264]:
wage$wage_ordinal=cut(wage$wage, breaks = c(0,13.461,19.231,27.778,528.846),
                      include.lowest = T, ordered_result = T,
                      labels=c("Muy bajo", "Bajo", "Alto","Muy alto"))
table(wage$wage_ordinal)


Muy bajo     Bajo     Alto Muy alto 
    1269     1462     1132     1287 

4.3 Creamos dos subsets que solo contengan a uno de los sexos

In [265]:
wage_only_women=wage%>%filter(sex == 1)
wage_only_men= wage%>%filter(sex == 0)
wage_only_women$case<- c(1)
wage_only_men$case<- c(1)

4.4 Agrupamos

In [266]:
wage_only_women%>%group_by(clg,wage_ordinal)%>%summarise(Casos=sum(case))

`summarise()` has grouped output by 'clg'. You can override using the `.groups`
argument.


clg,wage_ordinal,Casos
<dbl>,<ord>,<dbl>
0,Muy bajo,484
0,Bajo,434
0,Alto,281
0,Muy alto,295
1,Muy bajo,107
1,Bajo,227
1,Alto,224
1,Muy alto,237



Existen 237 mujeres con un college dregee que poseen un salario del 25% más rico de la muestra.

In [267]:
wage_only_women%>%group_by(ad,wage_ordinal)%>%summarise(Casos=sum(case))

`summarise()` has grouped output by 'ad'. You can override using the `.groups`
argument.


ad,wage_ordinal,Casos
<dbl>,<ord>,<dbl>
0,Muy bajo,558
0,Bajo,581
0,Alto,392
0,Muy alto,357
1,Muy bajo,33
1,Bajo,80
1,Alto,113
1,Muy alto,175


Existen 175 mujeres con un advanced degree que poseen un salario del 25% más rico de la muestra. En ambos dataframes, las mujeres que no poseen alguno de estos grados académicos tienden a ser más propensas a tener un salario del 25% menos rico de la muestra, en comparación a las mujeres que sí poseen un mayor nivel académico. Ello nos lleva a inferir que probablemente el nivel educativo puede influir en la obtención de mejores salarios.

5. Obtener el número de hombres con un grado académico medio-bajo que poseen un salario perteneciente al 25% más rico de la muestra

In [268]:
wage_only_men%>%group_by(hsg, wage_ordinal)%>%summarise(Casos=sum(case))

`summarise()` has grouped output by 'hsg'. You can override using the `.groups`
argument.


hsg,wage_ordinal,Casos
<dbl>,<ord>,<dbl>
0,Muy bajo,369
0,Bajo,525
0,Alto,477
0,Muy alto,648
1,Muy bajo,309
1,Bajo,276
1,Alto,150
1,Muy alto,107


Existen 107 trabajadores hombres con secundaria completa que poseen un salario perteneciente al 25% más rico de la muestra.

In [269]:
wage_only_men%>%group_by(shs, wage_ordinal)%>%summarise(Casos=sum(case))

`summarise()` has grouped output by 'shs'. You can override using the `.groups`
argument.


shs,wage_ordinal,Casos
<dbl>,<ord>,<dbl>
0,Muy bajo,634
0,Bajo,775
0,Alto,614
0,Muy alto,747
1,Muy bajo,44
1,Bajo,26
1,Alto,13
1,Muy alto,8


Existen solo 8 trabajadores hombres con secundaria incompleta que tienen un salario perteneciente al 25% más rico de la muestra. Para ambos casos, los hombres que no poseen alguno de estos grados educativos son más propensos a poseer un salario pertenenciente al 25% más pobre. Si comparamos estos resultados con el anterior grupo, podemos observar que hay más mujeres con un salario altísimo en comparación con los hombres, pero ello puede deberse al mayor nivel educativo que poseen.

6. Creación de dos bases de datos

In [270]:
Y=wage%>%select(3)

In [271]:
X=wage%>%select(!(2:3))

8. Hacer tres matrices para nuestros predictores 

In [272]:
names(X)

[1] "rownames"     "sex"          "shs"          "hsg"          "scl"         
 [6] "clg"          "ad"           "mw"           "so"           "we"          
[11] "ne"           "exp1"         "exp2"         "exp3"         "exp4"        
[16] "occ"          "occ2"         "ind"          "ind2"         "wage_ordinal"

Primero convertimos en dummy las variables "occ2" y "ind2"

In [273]:
X<- fastDummies::dummy_cols(X, select_columns = "occ2",remove_first_dummy = TRUE)
X<- fastDummies::dummy_cols(X, select_columns = "ind2",remove_first_dummy = TRUE)

In [274]:
names(X)

[1] "rownames"     "sex"          "shs"          "hsg"          "scl"         
 [6] "clg"          "ad"           "mw"           "so"           "we"          
[11] "ne"           "exp1"         "exp2"         "exp3"         "exp4"        
[16] "occ"          "occ2"         "ind"          "ind2"         "wage_ordinal"
[21] "occ2_2"       "occ2_3"       "occ2_4"       "occ2_5"       "occ2_6"      
[26] "occ2_7"       "occ2_8"       "occ2_9"       "occ2_10"      "occ2_11"     
[31] "occ2_12"      "occ2_13"      "occ2_14"      "occ2_15"      "occ2_16"     
[36] "occ2_17"      "occ2_18"      "occ2_19"      "occ2_20"      "occ2_21"     
[41] "occ2_22"      "ind2_3"       "ind2_4"       "ind2_5"       "ind2_6"      
[46] "ind2_7"       "ind2_8"       "ind2_9"       "ind2_10"      "ind2_11"     
[51] "ind2_12"      "ind2_13"      "ind2_14"      "ind2_15"      "ind2_16"     
[56] "ind2_17"      "ind2_18"      "ind2_19"      "ind2_20"      "ind2_21"     
[61] "ind2_22"

8.1

In [275]:
X_1=X%>%select(!c(mw,wage_ordinal,ind,rownames,exp2,exp3,exp4))

8.2

In [276]:
X_2=X%>%select(!c(mw,wage_ordinal,ind,rownames))

In [277]:
experience<- X%>%select(exp1,exp2,exp3,exp4)

In [278]:
names(X)

[1] "rownames"     "sex"          "shs"          "hsg"          "scl"         
 [6] "clg"          "ad"           "mw"           "so"           "we"          
[11] "ne"           "exp1"         "exp2"         "exp3"         "exp4"        
[16] "occ"          "occ2"         "ind"          "ind2"         "wage_ordinal"
[21] "occ2_2"       "occ2_3"       "occ2_4"       "occ2_5"       "occ2_6"      
[26] "occ2_7"       "occ2_8"       "occ2_9"       "occ2_10"      "occ2_11"     
[31] "occ2_12"      "occ2_13"      "occ2_14"      "occ2_15"      "occ2_16"     
[36] "occ2_17"      "occ2_18"      "occ2_19"      "occ2_20"      "occ2_21"     
[41] "occ2_22"      "ind2_3"       "ind2_4"       "ind2_5"       "ind2_6"      
[46] "ind2_7"       "ind2_8"       "ind2_9"       "ind2_10"      "ind2_11"     
[51] "ind2_12"      "ind2_13"      "ind2_14"      "ind2_15"      "ind2_16"     
[56] "ind2_17"      "ind2_18"      "ind2_19"      "ind2_20"      "ind2_21"     
[61] "ind2_22"

In [279]:
experience_2<- X%>%select(!c(mw,wage_ordinal,ind,rownames,sex,exp1,exp2,exp3,exp4))

In [280]:
for (exp in experience) {
    new_df=exp*experience_2
    print(new_df)
}

      shs hsg  scl clg ad so we   ne       occ  occ2  ind2 occ2_2 occ2_3 occ2_4
1     0.0   0  0.0   7  0  0  0  7.0   25200.0  77.0 126.0    0.0    0.0    0.0
2     0.0   0  0.0  31  0  0  0 31.0   94550.0 310.0 279.0    0.0    0.0    0.0
3     0.0  18  0.0   0  0  0  0 18.0  112680.0 342.0  72.0    0.0    0.0    0.0
4     0.0   0  0.0   0 25  0  0 25.0   10500.0  25.0 300.0    0.0    0.0    0.0
5     0.0   0  0.0  22  0  0  0 22.0   44330.0 132.0 484.0    0.0    0.0    0.0
6     0.0   0  0.0   1  0  0  0  1.0    1650.0   5.0  14.0    0.0    0.0    0.0
7     0.0  42  0.0   0  0  0  0 42.0  215040.0 714.0 588.0    0.0    0.0    0.0
8     0.0  37  0.0   0  0  0  0 37.0  193880.0 629.0 333.0    0.0    0.0    0.0
9     0.0  31  0.0   0  0  0  0 31.0  125240.0 403.0 589.0    0.0    0.0    0.0
10    0.0   0  0.0   4  0  0  0  4.0   13020.0  40.0  72.0    0.0    0.0    0.0
11    0.0   7  0.0   0  0  0  0  7.0   28140.0  91.0 126.0    0.0    0.0    0.0
12    0.0  30  0.0   0  0  0  0 30.0  12

In [281]:
X_2.1<-data.frame(X_2,new_df)

8.3

In [282]:
X_3<-data.frame(X_2['sex'],(X_2%>%select(!(sex)))^2)

# 3. Linear Regressions 

9. 

In [283]:
data_1<- data.frame(Y,X_1)
data_2<- data.frame(Y,X_2.1)
data_3<- data.frame(Y,X_3)

In [284]:
sample_1 <- sample.split(data_1$lwage, SplitRatio = 0.80)
train_1 <- subset(data_1, sample_1 == TRUE)
test_1 <- subset(data_1, sample_1 == FALSE)

sample_2 <- sample.split(data_2$lwage, SplitRatio = 0.80)
train_2 <- subset(data_2, sample_2 == TRUE)
test_2 <- subset(data_2, sample_2 == FALSE)

sample_3 <- sample.split(data_3$lwage, SplitRatio = 0.80)
train_3 <- subset(data_3, sample_3 == TRUE)
test_3 <- subset(data_3, sample_3 == FALSE)

10. Estimando los modelos 

In [285]:
modelo_1<-lm(lwage~., data=train_1)
modelo_2<- lm(lwage~., data=train_2)
modelo_3<- lm(lwage~., data=train_3)

Estimamos la variable dependiente predicha por el modelo en grupo de prueba

In [286]:
y_1_test=test_1$lwage
y_2_test=test_2$lwage
y_3_test=test_3$lwage


y_1_test_pred = predict(modelo_1, test_1%>%select(!(lwage)))
y_2_test_pred = predict(modelo_2, test_2%>%select(!(lwage)))
y_3_test_pred = predict(modelo_3, test_3%>%select(!(lwage)))

Warning message in predict.lm(modelo_3, test_3 %>% select(!(lwage))):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"


In [287]:
mse_1_test=mean((y_1_test - y_1_test_pred)^2)
mse_2_test=mean((y_2_test - y_2_test_pred)^2)
mse_3_test=mean((y_3_test - y_3_test_pred)^2)

print(paste0("MSE out of sample of basic model is ", mse_1_test))
print(paste0("MSE out of sample of flexible model is ", mse_2_test))
print(paste0("MSE out of sample of more flexible model is ", mse_3_test))

R_sq_1_test= 1 - mse_1_test / mean((y_1_test - mean(y_1_test))^2)
R_sq_2_test = 1 - mse_2_test / mean((y_2_test - mean(y_2_test))^2)
R_sq_3_test = 1 - mse_3_test / mean((y_3_test - mean(y_3_test))^2)

print(paste0("R-squared out of sample of basic model is ", R_sq_1_test))
print(paste0("R-squared out of sample of flexible model is ", R_sq_2_test))
print(paste0("R-squared out of sample of more flexible model is ", R_sq_3_test))

adj_mse_1_test = 1030 / (1030-55) * mse_1_test
adjR_sq_1_test = 1 - adj_mse_1_test / mean((y_1_test-mean(y_1_test))^2)

adj_mse_2_test = 1030 / (1030-110) * mse_2_test
adjR_sq_2_test = 1 - adj_mse_2_test / mean((y_2_test-mean(y_2_test))^2)

adj_mse_3_test = 1030 / (1030-58) * mse_3_test
adjR_sq_3_test = 1 - adj_mse_3_test / mean((y_3_test-mean(y_3_test))^2)

print(paste0("Adjusted R-squared out of sample of basic model  is ", adjR_sq_1_test))
print(paste0("Adjusted R-squared out of sample of flexible model  is ", adjR_sq_2_test))
print(paste0("Adjusted R-squared out of sample of more flexible model is ", adjR_sq_3_test))

[1] "MSE out of sample of basic model is 0.202993200097623"
[1] "MSE out of sample of flexible model is 0.188199615413922"
[1] "MSE out of sample of more flexible model is 0.193945961775633"
[1] "R-squared out of sample of basic model is 0.255451193904896"
[1] "R-squared out of sample of flexible model is 0.309711857852355"
[1] "R-squared out of sample of more flexible model is 0.288635115769549"
[1] "Adjusted R-squared out of sample of basic model  is 0.213451004843121"
[1] "Adjusted R-squared out of sample of flexible model  is 0.227177406073832"
[1] "Adjusted R-squared out of sample of more flexible model is 0.246187416916292"


In [288]:
y_1_train=train_1$lwage
y_2_train=train_2$lwage
y_3_train=train_3$lwage

y_1_train_pred = predict(modelo_1, train_1%>%select(!(lwage)))
y_2_train_pred = predict(modelo_2, train_2%>%select(!(lwage)))
y_3_train_pred = predict(modelo_3, train_3%>%select(!(lwage)))

Warning message in predict.lm(modelo_3, train_3 %>% select(!(lwage))):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"


In [289]:
mse_1_train=mean((y_1_train - y_1_train_pred)^2)
mse_2_train=mean((y_2_train - y_2_train_pred)^2)
mse_3_train=mean((y_3_train - y_3_train_pred)^2)

print(paste0("MSE in the sample of basic model is ", mse_1_train))
print(paste0("MSE in the sample of flexible model is ", mse_2_train))
print(paste0("MSE in the sample of more flexible model is ", mse_3_train))

R_sq_1_train= 1 - mse_1_train / mean((y_1_train - mean(y_1_train))^2)
R_sq_2_train = 1 - mse_2_train / mean((y_2_train - mean(y_2_train))^2)
R_sq_3_train = 1 - mse_3_train / mean((y_3_train - mean(y_3_train))^2)

print(paste0("R-squared in the sample of basic model is ", R_sq_1_train))
print(paste0("R-squared in the sample of flexible model is ", R_sq_2_train))
print(paste0("R-squared in the sample of more flexible model is ", R_sq_3_train))

adj_mse_1_train = 1030 / (1030-55) * mse_1_train
adjR_sq_1_train = 1 - adj_mse_1_test / mean((y_1_train-mean(y_1_train))^2)

adj_mse_2_train = 1030 / (1030-110) * mse_2_train
adjR_sq_2_train = 1 - adj_mse_2_train / mean((y_2_train-mean(y_2_train))^2)

adj_mse_3_train = 1030 / (1030-58) * mse_3_train
adjR_sq_3_train = 1 - adj_mse_3_train / mean((y_3_train-mean(y_3_train))^2)

print(paste0("Adjusted R-squared in the sample of basic model  is ", adjR_sq_1_train))
print(paste0("Adjusted R-squared in the sample of flexible model  is ", adjR_sq_2_train))
print(paste0("Adjusted R-squared in the sample of more flexible model is ", adjR_sq_3_train))

[1] "MSE in the sample of basic model is 0.229287076577505"
[1] "MSE in the sample of flexible model is 0.22615859438363"
[1] "MSE in the sample of more flexible model is 0.229795374938077"
[1] "R-squared in the sample of basic model is 0.316769316641703"
[1] "R-squared in the sample of flexible model is 0.326091582244722"
[1] "R-squared in the sample of more flexible model is 0.315254686853461"
[1] "Adjusted R-squared in the sample of basic model  is 0.360998490663008"
[1] "Adjusted R-squared in the sample of flexible model  is 0.245515575773982"
[1] "Adjusted R-squared in the sample of more flexible model is 0.274395398620438"


Interpretación: 
- En términos generales, los tres modelos presentan métricas de predicción más altas en el conjunto de entrenamiento, en comparación cuando se evalúan en el conjunto de prueba. 
- En el conjunto de entrenamiento, se puede observar que el modelo flexible tiene un mayor r cuadrado(0.327) en comparación con el modelo básico y el modelo más flexible. No obstante, cuando utilizamos el r cuadrado ajustado, el cual incluye una penalidad por la inclusión de variables irrelevantes, podemos observar que el modelo básico es el que posee la mayor capacidad predictiva (0.361). 
- Los resultados de evaluación en el conjunto de entrenamiento no se pueden replicar en el conjunto de prueba, pues allí el modelo que posee una mayor capacidad predictiva es el modelo más flexible (0.246)
- Se puede identificar un  ligero caso de overfitting con el segundo modelo, el que posee  el mayor número de variables (110), debido a que cuando se evalúa su capacidad predictiva en el conjunto de prueba, sus indicadores de evaluación decaen ligeramente (MSE_Train: 0.226, MSE_Test: 0.1881), (R2_Train: 0.326, R2_Test: 0.310), (Rajustado_Train: 0.245, Rajustado_Test: 0.227)